# Factor Pricing ML — Exploration

Pipeline complet : données -> baseline OLS -> modèles ML -> comparaison hors échantillon -> backtest.

In [1]:
import sys
sys.path.append('../src')

import pandas as pd
import matplotlib.pyplot as plt

from data_loader import load_fama_french_5factors, load_asset_returns
#from models import fit_ols_baseline, fit_ml_models, evaluate_models, time_series_split
from backtest import compare_to_buy_and_hold

%matplotlib inline

## 1. Chargement des données

In [3]:
ff5 = load_fama_french_5factors()
ff5

,Mkt-RF,SMB,HML,RMW,CMA,RF
date,,,,,,
1963-07-01,-0.0039,-0.0048,-0.0084,0.0064,-0.0115,0.0027
1963-08-01,0.0508,-0.0080,0.0172,0.0040,-0.0038,0.0025
1963-09-01,-0.0157,-0.0043,-0.0002,-0.0078,0.0015,0.0027
1963-10-01,0.0254,-0.0134,-0.0003,0.0279,-0.0225,0.0029
1963-11-01,-0.0086,-0.0085,0.0178,-0.0043,0.0227,0.0027
...,...,...,...,...,...,...
2026-03-01,-0.0518,0.0067,0.0329,-0.0203,-0.0012,0.0029
2026-04-01,0.0995,0.0046,-0.0137,-0.0415,-0.0380,0.0029
2026-05-01,0.0491,-0.0265,-0.0231,-0.0816,-0.0146,0.0031


In [4]:
# Actif test : S&P 500 (à remplacer par un portefeuille trié si tu veux aller plus loin)
asset = load_asset_returns(["^GSPC"], start="2000-01-01")
asset.columns = ["SP500"]
asset.tail()

[*********************100%***********************]  1 of 1 completed


,SP500
Date,
2026-05-31,0.051470
2026-06-30,-0.010646
2026-07-31,-0.001285
2026-08-31,0.026225
2026-09-30,0.007451


## 2. Préparation des données (aligner les dates, calculer le rendement excédentaire)

In [5]:
df = ff5.join(asset, how="inner")
df["excess_ret"] = df["SP500"] - df["RF"]
df = df.dropna()

X = df[["Mkt-RF", "SMB", "HML", "RMW", "CMA"]]
y = df["excess_ret"]

X_train, X_test, y_train, y_test = time_series_split(X, y, test_size=0.2)
print(f"Train: {len(X_train)} obs, Test: {len(X_test)} obs")

NameError: name 'time_series_split' is not defined

## 3. Baseline OLS

In [ ]:
ols_model = fit_ols_baseline(y_train, X_train)
print(ols_model.summary())

## 4. Modèles ML

In [ ]:
ml_models = fit_ml_models(X_train, y_train)
results = evaluate_models(ml_models, X_test, y_test, ols_model=ols_model, X_test_ols=X_test)
results

## 5. Backtest : la meilleure stratégie ML bat-elle le buy-and-hold ?

In [ ]:
best_model_name = results.iloc[0]["model"]
if best_model_name in ml_models:
    best_preds = pd.Series(ml_models[best_model_name].predict(X_test), index=X_test.index)
    summary = compare_to_buy_and_hold(best_preds, y_test)
    display(summary)
else:
    print("Le meilleur modèle est l'OLS — refais tourner avec ses prédictions si besoin.")

## Prochaines étapes

- Étendre à d'autres actifs / portefeuilles triés (taille, valeur)
- Ajouter du feature engineering (rendements décalés, volatilité réalisée)
- Analyse d'importance des facteurs (SHAP) pour interpréter les modèles ML
- Tester la robustesse sur différentes périodes (crise 2008, COVID, etc.)